# Camada Silver

In [0]:
%sql

-- 1. Limpeza de Geolocalização e Dissolução na tabela Customers

CREATE OR REPLACE TABLE projeto_olist.olist_silver.olist_customers_silver AS
WITH cte_geo_clean AS(

    SELECT 
        geolocation_zip_code_prefix,
        FIRST(geolocation_city) AS geo_city,
        FIRST(geolocation_state) AS geo_state
    FROM projeto_olist.olist_bronze.olist_geolocation_dataset_raw
    GROUP BY geolocation_zip_code_prefix
)

SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    COALESCE(g.geo_city, c.customer_city) AS customer_city,
    COALESCE(g.geo_state, c.customer_state) AS customer_state
FROM projeto_olist.olist_bronze.olist_customers_dataset_raw c
LEFT JOIN cte_geo_clean g
    ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix;

-- 2. Dissolução da geolocalização em Sellers(Vendedores)

CREATE OR REPLACE TABLE projeto_olist.olist_silver.olist_sellers_silver AS
WITH cte_geo_clean AS(
    SELECT 
        geolocation_zip_code_prefix, 
        FIRST(geolocation_city) as geo_city,
        FIRST(geolocation_state) as geo_state
    FROM projeto_olist.olist_bronze.olist_geolocation_dataset_raw
    GROUP BY geolocation_zip_code_prefix

)

SELECT 
    s.seller_id,
    s.seller_zip_code_prefix,
    COALESCE(g.geo_city, s.seller_city) AS seller_city,
    COALESCE(g.geo_state, s.seller_state) AS seller_state
FROM projeto_olist.olist_bronze.olist_sellers_dataset_raw s
LEFT JOIN cte_geo_clean g 
    ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix;

-- 3. Tratamento de Datas(Texto para TimeStamp)

CREATE OR REPLACE TABLE projeto_olist.olist_silver.olist_orders_silver AS 
SELECT 
    order_id,
    customer_id,
    order_status, 
    -- Converte as string em formato data/hora
    CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
    CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
FROM projeto_olist.olist_bronze.olist_orders_dataset_raw;

-- 4. Tratando produtos

CREATE OR REPLACE TABLE projeto_olist.olist_silver.olist_products_silver AS 
SELECT 
    product_id,
    COALESCE(product_category_name,'Não Informado') AS product_category_name,
    CAST(product_weight_g AS INT) AS product_weight_g,
    CAST(product_length_cm AS INT) AS product_lenght_cm,
    CAST(product_height_cm AS INT) AS product_height_cm,
    CAST(product_width_cm AS INT) AS product_width_cm 
FROM projeto_olist.olist_bronze.olist_products_dataset_raw;

-- 5. Tartando tipagem dos valores de pedidos

CREATE OR REPLACE TABLE projeto_olist.olist_silver.olist_orders_items_silver AS 
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
    CAST(price AS DECIMAL(10,2)) AS Price,
    CAST(freight_value AS DECIMAL(10,2)) AS freight_value
FROM projeto_olist.olist_bronze.olist_order_items_dataset_raw


